# Transferability of LLM-generated adversarial examples across BERT-family models

Code and results for the HW2 draft. The notebook compares a word-level attack (TextFooler) with iterative LLM sentence rewriting on SST-2, and measures how often an adversarial example built against one classifier also fools the other two.

**Pipeline**
1. Select SST-2 validation sentences that BERT-base, RoBERTa-base and DistilBERT all classify correctly.
2. Attack each classifier (the *source*) with both methods.
3. Count an attack as successful if it fools the source, has sentence-embedding similarity of at least 0.80 to the original, and keeps the original sentiment according to an LLM judge.
4. Test every successful example on the other two classifiers (*transfer*) and compare the methods with bootstrap confidence intervals.
5. Check the sentiment criterion with an NLI model and a blind, hand-labeled sample.

## Environment
- Kaggle notebook with Internet and GPU enabled (Tesla T4, printed in Section 1.2), Python 3.13.
- Packages are installed without version pins (Section 1.1), so the exact versions used in this run were not recorded.
- Models from Hugging Face: `textattack/bert-base-uncased-SST-2`, `textattack/roberta-base-SST-2`, `distilbert/distilbert-base-uncased-finetuned-sst-2-english` (classifiers); `Qwen/Qwen2.5-3B-Instruct` (rewriting and sentiment judge); `sentence-transformers/all-MiniLM-L6-v2` (similarity); `roberta-large-mnli` (NLI check).
- Data: GLUE SST-2 via `load_dataset('glue', 'sst2')`.

## Running the notebook
Run the cells from top to bottom, restarting the session after the install cell if the platform asks. All settings are in Section 1.2; `mode = 'main'` uses 200 validation sentences.

Attack results are appended to disk after every sentence, and finished sentences are skipped on a re-run, so an interrupted run can be resumed. Files are written to `/kaggle/working/hw2_transfer_distilbert_fixed/`:
`main_textfooler.jsonl`, `main_llm_v3.jsonl`, `judge_cache.json`, `main_transfer_table.csv`, `handcheck.csv`, `handcheck_blind.csv`, `handcheck_key.csv`. This folder is temporary storage on Kaggle, so download it before the session ends.

LLM sampling is seeded, but sampled text can still differ across hardware or library versions. `main_llm_v3.jsonl` contains the rewrites behind every reported number.

The outputs stored in this file come from a single run in `main` mode. The hand-labeling cell (Section 6.1) is interactive and stores no output.

## Where each result comes from
| Result | Section |
|---|---|
| Classifier clean accuracy and confusion matrices | 2.1 |
| Judge agreement with gold labels on the original sentences | 3.2 |
| Source success rates, effect of each validity check, queries per success | 5.1 |
| Per-pair transfer rates, confidence intervals, unconditional rates (`main_transfer_table.csv`) | 5.2 |
| Edit-size statistics and transfer by edit size | 5.3 |
| Pooled transfer rate and similarity-threshold sensitivity | 5.4 |
| NLI check and NLI-filtered comparison | 6.2, 6.3 |
| Hand-check sampling, labeling and results | 5.3, 6.1, 6.4 |

## Random seeds
| Component | Seed |
|---|---|
| Global (`random`, NumPy, PyTorch) | 42 (`CFG['seed']`) |
| Sentence selection | `RandomState(42)` |
| LLM sampling | 42 + sentence id, set before each (sentence, source) attack |
| Bootstrap, per-pair table (5.2) | `RandomState(0)`, 2000 resamples |
| Bootstrap, pooled and NLI-subset comparisons (5.4, 6.3) | `RandomState(1)`, 2000 resamples |
| Hand-check sample and shuffle (5.3) | `random_state=0` |
| Random examples printed (5.1, 5.3) | `random_state=0`, `random_state=1` |

## 1. Setup
### 1.1 Install packages
Installs TextAttack and the Hugging Face libraries. Restart the session afterwards if the platform asks for it. No output of interest.

In [ ]:
!pip -q install textattack transformers accelerate sentence-transformers datasets

### 1.2 Configuration and random seeds
Paths, experiment settings and the global seed. Every later cell reads `CFG`.

| Setting | Value | Meaning |
|---|---|---|
| `mode` | `main` | `main`: 200 validation sentences. `pilot`: 30 training sentences, used only while developing |
| `min_words` | 8 | minimum sentence length |
| `seed` | 42 | global seed |
| `llm_name` | Qwen2.5-3B-Instruct | rewriting model and sentiment judge |
| `llm_tag` | `v3` | suffix of the LLM result file |
| `n_candidates` x `max_rounds` | 5 x 10 | rewrites per LLM call x maximum calls per sentence (at most 50 classifier queries) |
| `temperature`, `top_p`, `max_new_tokens` | 0.8, 0.95, 80 | sampling settings |
| `sim_model` | all-MiniLM-L6-v2 | sentence encoder used for similarity |
| `sim_threshold` | 0.80 | minimum similarity to the original, identical for both methods |

**Output:** the mode and the GPU name.

In [1]:
import os, json, random, time, difflib
import numpy as np, pandas as pd, torch

OUT = '/kaggle/working/hw2_transfer_distilbert_fixed'
os.makedirs(OUT, exist_ok=True)

CFG = dict(
    mode='main',
    n_sentences={'pilot': 30, 'main': 200},
    min_words=8,
    seed=42,
    llm_name='Qwen/Qwen2.5-3B-Instruct',
    llm_tag='v3',
    n_candidates=5,
    max_rounds=10,
    temperature=0.8, top_p=0.95, max_new_tokens=80,
    sim_model='sentence-transformers/all-MiniLM-L6-v2',
    sim_threshold=0.80,
)
CLASSIFIERS = {
    'BERT': 'textattack/bert-base-uncased-SST-2',
    'RoBERTa': 'textattack/roberta-base-SST-2',
    'DistilBERT': 'distilbert/distilbert-base-uncased-finetuned-sst-2-english',
}
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(CFG['seed'])
dev = 'cuda'
print(CFG['mode'], torch.cuda.get_device_name(0))

main Tesla T4


### 1.3 Load the classifiers and define helpers
Loads the three SST-2 classifiers and defines `probs` (class probabilities for a list of texts) and the functions that read and append to the result files.

**Output:** checkpoint names and label mappings.

In [2]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
CLF = {}
for k, name in CLASSIFIERS.items():
    tok = AutoTokenizer.from_pretrained(name)
    mdl = AutoModelForSequenceClassification.from_pretrained(name).to(dev).eval()
    CLF[k] = (tok, mdl)
    print(k, 'checkpoint:', name)
    print('  id2label:', mdl.config.id2label)
    print('  label2id:', mdl.config.label2id)

@torch.no_grad()
def probs(name, texts, bs=64):
    tok, mdl = CLF[name]
    out = []
    for i in range(0, len(texts), bs):
        enc = tok(texts[i:i+bs], padding=True, truncation=True, max_length=128,
                  return_tensors='pt').to(dev)
        out.append(torch.softmax(mdl(**enc).logits, -1).cpu())
    return torch.cat(out).numpy()

def path(method):
    suffix = ('_' + CFG['llm_tag']) if method == 'llm' else ''
    return f"{OUT}/{CFG['mode']}_{method}{suffix}.jsonl"
def load(method):
    return pd.DataFrame([json.loads(l) for l in open(path(method))])
def load_done(method):
    if not os.path.exists(path(method)): return set()
    return {(r['source'], r['sid']) for r in map(json.loads, open(path(method)))}
def append(method, rec):
    with open(path(method), 'a') as f: f.write(json.dumps(rec) + '\n')

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/477 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

I0000 00:00:1794286331.373442     256 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

... (11 outputs left out)

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

DistilBERT checkpoint: distilbert/distilbert-base-uncased-finetuned-sst-2-english
  id2label: {0: 'NEGATIVE', 1: 'POSITIVE'}
  label2id: {'NEGATIVE': 0, 'POSITIVE': 1}


## 2. Data
### 2.1 Clean accuracy and sentence selection
Evaluates each classifier on the SST-2 validation set and stops if any accuracy is below 0.85. Then selects the attack sentences: at least `min_words` words, classified correctly by all three models, shuffled with the global seed.

**Output:** accuracy, confusion matrix and classification report per classifier, and the number of selected sentences.

In [3]:
from datasets import load_dataset
from sklearn.metrics import confusion_matrix, classification_report

sst = load_dataset('glue', 'sst2')
v_text = [s.strip() for s in sst['validation']['sentence']]
v_lab = np.array(sst['validation']['label'])  # 0 = negative, 1 = positive

clean_acc = {}
clean_pred = {}

for k, (tok, mdl) in CLF.items():
    p = probs(k, v_text)
    pred = p.argmax(1)
    clean_pred[k] = pred
    clean_acc[k] = float((pred == v_lab).mean())

    print(f'\\n{k}')
    print('  checkpoint:', CLASSIFIERS[k])
    print('  id2label:', mdl.config.id2label)
    print('  label2id:', mdl.config.label2id)
    print('  clean accuracy:', round(clean_acc[k], 4))
    print('  predicted class counts [0, 1]:', np.bincount(pred, minlength=2).tolist())
    print('  gold class counts [0, 1]:     ', np.bincount(v_lab, minlength=2).tolist())
    print('  confusion matrix (rows=gold, columns=pred):')
    print(confusion_matrix(v_lab, pred, labels=[0, 1]))
    print(classification_report(v_lab, pred, labels=[0, 1],
                                target_names=['negative', 'positive'], digits=4))

for k, acc in clean_acc.items():
    if acc < 0.85:
        raise RuntimeError(
            f'{k} clean accuracy is only {acc:.3%}. Stop here and inspect '
            'the checkpoint, label mapping, tokenizer, and predictions before '
            'running attacks.'
        )

def select(split, n):
    ds = sst[split]
    sents = [s.strip() for s in ds['sentence']]
    labs = np.array(ds['label'])
    rng = np.random.RandomState(CFG['seed'])
    idx = rng.permutation(len(sents))[:3000] if split == 'train' else np.arange(len(sents))
    t = [sents[i] for i in idx]; y = labs[idx]
    ok = np.array([len(s.split()) >= CFG['min_words'] for s in t])
    for k in CLF:
        ok &= (probs(k, t).argmax(1) == y)
    cand = np.where(ok)[0]
    rng.shuffle(cand)
    return [dict(sid=int(idx[i]), text=t[i], label=int(y[i])) for i in cand[:n]]

ITEMS = select('train' if CFG['mode'] == 'pilot' else 'validation',
               CFG['n_sentences'][CFG['mode']])
print('\\nSelected', len(ITEMS), 'sentences; example:', ITEMS[0] if ITEMS else 'NONE')
if len(ITEMS) == 0:
    raise RuntimeError('No eligible examples were selected. Check model predictions and filtering.')

README.md: 0.00B [00:00, ?B/s]

sst2/train-00000-of-00001.parquet:   0%|          | 0.00/3.11M [00:00<?, ?B/s]

sst2/validation-00000-of-00001.parquet:   0%|          | 0.00/72.8k [00:00<?, ?B/s]

sst2/test-00000-of-00001.parquet:   0%|          | 0.00/148k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

\nBERT
  checkpoint: textattack/bert-base-uncased-SST-2
  id2label: {0: 'LABEL_0', 1: 'LABEL_1'}
  label2id: {'LABEL_0': 0, 'LABEL_1': 1}
  clean accuracy: 0.9243
  predicted class counts [0, 1]: [420, 452]
  gold class counts [0, 1]:      [428, 444]
  confusion matrix (rows=gold, columns=pred):
[[391  37]
 [ 29 415]]
              precision    recall  f1-score   support

    negative     0.9310    0.9136    0.9222       428
    positive     0.9181    0.9347    0.9263       444

    accuracy                         0.9243       872
   macro avg     0.9245    0.9241    0.9243       872
weighted avg     0.9244    0.9243    0.9243       872

\nRoBERTa
  checkpoint: textattack/roberta-base-SST-2
  id2label: {0: 'LABEL_0', 1: 'LABEL_1'}
  label2id: {'LABEL_0': 0, 'LABEL_1': 1}
  clean accuracy: 0.9438
  predicted class counts [0, 1]: [429, 443]
  gold class counts [0, 1]:      [428, 444]
  confusion matrix (rows=gold, columns=pred):
[[404  24]
 [ 25 419]]
              precision    recall  

## 3. Scoring components
### 3.1 Similarity scorer and result records
`sim` returns the cosine similarity between a sentence and a list of rewrites (MiniLM embeddings). `finish` builds one result record and, when the source classifier was fooled, also tests the example on the other two classifiers.

No printed output.

In [4]:
from sentence_transformers import SentenceTransformer
SIM = SentenceTransformer(CFG['sim_model'], device=dev)
def sim(a, bs):
    ea = SIM.encode([a], convert_to_tensor=True, normalize_embeddings=True)
    eb = SIM.encode(bs, convert_to_tensor=True, normalize_embeddings=True)
    return (eb @ ea.T).squeeze(1).cpu().tolist()

def finish(method, source, it, adv, fooled, queries, s, **extra):
    rec = dict(method=method, source=source, sid=it['sid'], text=it['text'], label=it['label'],
               adv=adv, fooled_source=bool(fooled), sim=s, queries=int(queries), **extra)
    if fooled:
        for t in CLF:
            if t != source:
                rec['fool_' + t] = bool(probs(t, [adv])[0].argmax() != it['label'])
    return rec

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

### 3.2 LLM and sentiment judge
Loads Qwen2.5-3B-Instruct in fp16. `judge1` returns the model's zero-shot sentiment label for a sentence by comparing the logits of "positive" and "negative" for the first answer token. Judgments are cached in `judge_cache.json`. The judge is first run on the original sentences to check its agreement with the gold labels.

**Output:** judge agreement with the gold labels.

In [5]:
from transformers import AutoModelForCausalLM
ltok = AutoTokenizer.from_pretrained(CFG['llm_name'])
llm = AutoModelForCausalLM.from_pretrained(CFG['llm_name'], torch_dtype=torch.float16).to(dev).eval()
NAMES = {0: 'negative', 1: 'positive'}

POS_IDS = [ltok.encode(w)[0] for w in ['positive', 'Positive']]
NEG_IDS = [ltok.encode(w)[0] for w in ['negative', 'Negative']]
JCACHE_PATH = f'{OUT}/judge_cache.json'
JCACHE = json.load(open(JCACHE_PATH)) if os.path.exists(JCACHE_PATH) else {}

@torch.no_grad()
def judge1(t):
    if t in JCACHE: return JCACHE[t]
    msgs = [{'role': 'system', 'content': 'You are a careful sentiment annotator.'},
            {'role': 'user', 'content': f"Movie review sentence: {t}\n\nIs the reviewer's overall opinion "
                                        f"about the movie positive or negative? Answer with one word: positive or negative."}]
    text = ltok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    enc = ltok(text, return_tensors='pt').to(dev)
    lg = llm(**enc).logits[0, -1].float()
    r = int(torch.logsumexp(lg[POS_IDS], 0) > torch.logsumexp(lg[NEG_IDS], 0))
    JCACHE[t] = r
    return r
def save_judge(): json.dump(JCACHE, open(JCACHE_PATH, 'w'))

jacc = float(np.mean([judge1(it['text']) == it['label'] for it in ITEMS]))
save_judge()
print('Judge agreement with gold label on original sentences:', round(jacc, 3))

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


... (3 outputs left out)

model-00002-of-00002.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Judge agreement with gold label on original sentences: 0.96


## 4. Attacks
### 4.1 Baseline: TextFooler
Runs the TextAttack TextFooler recipe on every selected sentence, once per source classifier. Each result (adversarial text, success flag, similarity, query count, and whether it fools the other two classifiers) is appended to `main_textfooler.jsonl`. Sentences already in the file are skipped.

**Output:** progress counter.

In [6]:
import textattack
from textattack.attack_recipes import TextFoolerJin2019
from textattack.models.wrappers import HuggingFaceModelWrapper
from textattack.attack_results import SuccessfulAttackResult
import nltk
for r in ['averaged_perceptron_tagger_eng', 'averaged_perceptron_tagger',
          'punkt', 'punkt_tab', 'universal_tagset', 'stopwords', 'wordnet', 'omw-1.4']:
    nltk.download(r, quiet=True)

def run_textfooler():
    done = load_done('textfooler')
    for source in CLF:
        todo = [it for it in ITEMS if (source, it['sid']) not in done]
        if not todo: continue
        tok, mdl = CLF[source]
        attack = TextFoolerJin2019.build(HuggingFaceModelWrapper(mdl, tok))
        for n, it in enumerate(todo):
            res = attack.attack(it['text'], it['label'])
            ok = isinstance(res, SuccessfulAttackResult)
            adv = res.perturbed_text() if ok else it['text']
            s = sim(it['text'], [adv])[0] if ok else None
            append('textfooler', finish('textfooler', source, it, adv, ok, res.num_queries, s))
            if n % 10 == 0: print(source, n, '/', len(todo))
run_textfooler()

textattack: Updating TextAttack package dependencies.
textattack: Downloading NLTK required packages.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /usr/share/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package omw to /usr/share/nltk_data...
[nltk_data]   Package omw is already up-to-date!
[nltk_data] Downloading package universal_tagset to
[nltk_data]     /usr/share/nltk_data...
[nltk_data]   Package universal_tagset is already up-to-date!
[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
/usr/local/lib/python3.13/dist-packages/jieba/__init__.py:44: SyntaxWarning: in

BERT 0 / 200
BERT 10 / 200
BERT 20 / 200
BERT 30 / 200
BERT 40 / 200
BERT 50 / 200
BERT 60 / 200
BERT 70 / 200
BERT 80 / 200
BERT 90 / 200
BERT 100 / 200
BERT 110 / 200
BERT 120 / 200
BERT 130 / 200
BERT 140 / 200
BERT 150 / 200
BERT 160 / 200
BERT 170 / 200
BERT 180 / 200
BERT 190 / 200


textattack: Goal function <class 'textattack.goal_functions.classification.untargeted_classification.UntargetedClassification'> compatible with model RobertaForSequenceClassification.


RoBERTa 0 / 200
RoBERTa 10 / 200
RoBERTa 20 / 200
RoBERTa 30 / 200
RoBERTa 40 / 200
RoBERTa 50 / 200
RoBERTa 60 / 200
RoBERTa 70 / 200
RoBERTa 80 / 200
RoBERTa 90 / 200
RoBERTa 100 / 200
RoBERTa 110 / 200
RoBERTa 120 / 200
RoBERTa 130 / 200
RoBERTa 140 / 200
RoBERTa 150 / 200
RoBERTa 160 / 200
RoBERTa 170 / 200
RoBERTa 180 / 200
RoBERTa 190 / 200


textattack: Goal function <class 'textattack.goal_functions.classification.untargeted_classification.UntargetedClassification'> compatible with model DistilBertForSequenceClassification.


DistilBERT 0 / 200
DistilBERT 10 / 200
DistilBERT 20 / 200
DistilBERT 30 / 200
DistilBERT 40 / 200
DistilBERT 50 / 200
DistilBERT 60 / 200
DistilBERT 70 / 200
DistilBERT 80 / 200
DistilBERT 90 / 200
DistilBERT 100 / 200
DistilBERT 110 / 200
DistilBERT 120 / 200
DistilBERT 130 / 200
DistilBERT 140 / 200
DistilBERT 150 / 200
DistilBERT 160 / 200
DistilBERT 170 / 200
DistilBERT 180 / 200
DistilBERT 190 / 200


### 4.2 LLM iterative rewriting
For each sentence and source classifier, the LLM proposes 5 rewrites per round for up to 10 rounds. The source classifier scores every candidate. A round succeeds if a candidate fools the source, has similarity of at least 0.80 and keeps the sentiment according to the judge. Otherwise the best candidate, with the classifier's label and confidence, goes back into the next prompt. Results are appended to `main_llm_v3.jsonl`.

**Output:** progress counter.

In [7]:
BAD = ('classifier', 'automated', 'algorithm', 'misclassif', 'this task')  # drop rewrites that mention the task
SYSTEM = 'You help researchers test the robustness of sentiment classifiers by rewriting movie-review sentences.'

def make_prompt(orig, label, last=None, fb=None, note=None):
    lab = NAMES[label]
    msg = (f'Original sentence: {orig}\n\n'
           f'Rewrite it in different words. A human reader must still clearly judge it {lab} '
           f'(do not reverse or soften the opinion, and do not add negations that reverse it), '
           f'but the phrasing should be indirect, subtle or unusual enough that an automatic '
           f'sentiment classifier might misclassify it. '
           f'Write only a natural review sentence; never mention classifiers, systems, models or this task.')
    if last is not None:
        msg += (f'\n\nYour previous rewrite was: {last}\n'
                f'The classifier labeled it {NAMES[fb[0]]} with {fb[1]:.0%} confidence. ')
        if note == 'drifted':
            msg += 'It also drifted too far from the original meaning, so stay closer to it. '
        elif note == 'flipped':
            msg += f'But it reversed the sentiment, so a human would no longer judge it {lab}. Keep it {lab}. '
        msg += 'Write a better rewrite.'
    msg += '\nReply with the rewritten sentence only.'
    return [{'role': 'system', 'content': SYSTEM}, {'role': 'user', 'content': msg}]

def clean(o):
    return o.strip().split('\n')[0].strip().strip('"').strip()

@torch.no_grad()
def generate(messages):
    text = ltok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    enc = ltok(text, return_tensors='pt').to(dev)
    out = llm.generate(**enc, do_sample=True, temperature=CFG['temperature'], top_p=CFG['top_p'],
                       max_new_tokens=CFG['max_new_tokens'], num_return_sequences=CFG['n_candidates'],
                       pad_token_id=ltok.eos_token_id)
    return [clean(o) for o in ltok.batch_decode(out[:, enc['input_ids'].shape[1]:], skip_special_tokens=True)]

def llm_attack(source, it):
    thr, label, orig = CFG['sim_threshold'], it['label'], it['text']
    last = fb = note = None
    queries = calls = n_flipped = 0
    last_sim = None
    for r in range(CFG['max_rounds']):
        cands = generate(make_prompt(orig, label, last, fb, note)); calls += 1
        cands = [c for c in dict.fromkeys(cands) if c and c.lower() != orig.lower()]
        cands = [c for c in cands if not any(b in c.lower() for b in BAD)]
        if not cands: continue
        p = probs(source, cands); queries += len(cands)  # one classifier query per candidate
        s = sim(orig, cands)
        good, flipped = [], []
        for i in range(len(cands)):
            if p[i].argmax() != label and s[i] >= thr:
                (good if judge1(cands[i]) == label else flipped).append(i)
        n_flipped += len(flipped)
        if good:
            i = max(good, key=lambda j: s[j])
            return finish('llm', source, it, cands[i], True, queries, s[i],
                          llm_calls=calls, n_flipped=n_flipped, judge_ok=True)
        passing = [i for i in range(len(cands)) if s[i] >= thr and i not in flipped]
        if passing:
            i, note = min(passing, key=lambda j: p[j][label]), None
        elif flipped:
            i, note = flipped[0], 'flipped'
        else:
            i, note = max(range(len(cands)), key=lambda j: s[j]), 'drifted'
        last, fb, last_sim = cands[i], (int(p[i].argmax()), float(p[i].max())), s[i]
    return finish('llm', source, it, last or orig, False, queries, last_sim,
                  llm_calls=calls, n_flipped=n_flipped, judge_ok=False)

def run_llm():
    done = load_done('llm')
    for source in CLF:
        for n, it in enumerate(ITEMS):
            if (source, it['sid']) in done: continue
            set_seed(CFG['seed'] + it['sid'])  # seed per sentence
            append('llm', llm_attack(source, it))
            if n % 10 == 0: print(source, n, '/', len(ITEMS))
    save_judge()
run_llm()

BERT 0 / 200
BERT 10 / 200
BERT 20 / 200
BERT 30 / 200
BERT 40 / 200
BERT 50 / 200
BERT 60 / 200
BERT 70 / 200
BERT 80 / 200
BERT 90 / 200
BERT 100 / 200
BERT 110 / 200
BERT 120 / 200
BERT 130 / 200
BERT 140 / 200
BERT 150 / 200
BERT 160 / 200
BERT 170 / 200
BERT 180 / 200
BERT 190 / 200
RoBERTa 0 / 200
RoBERTa 10 / 200
RoBERTa 20 / 200
RoBERTa 30 / 200
RoBERTa 40 / 200
RoBERTa 50 / 200
RoBERTa 60 / 200
RoBERTa 70 / 200
RoBERTa 80 / 200
RoBERTa 90 / 200
RoBERTa 100 / 200
RoBERTa 110 / 200
RoBERTa 120 / 200
RoBERTa 130 / 200
RoBERTa 140 / 200
RoBERTa 150 / 200
RoBERTa 160 / 200
RoBERTa 170 / 200
RoBERTa 180 / 200
RoBERTa 190 / 200
DistilBERT 0 / 200
DistilBERT 10 / 200
DistilBERT 20 / 200
DistilBERT 30 / 200
DistilBERT 40 / 200
DistilBERT 50 / 200
DistilBERT 60 / 200
DistilBERT 70 / 200
DistilBERT 80 / 200
DistilBERT 90 / 200
DistilBERT 100 / 200
DistilBERT 110 / 200
DistilBERT 120 / 200
DistilBERT 130 / 200
DistilBERT 140 / 200
DistilBERT 150 / 200
DistilBERT 160 / 200
DistilBERT 170 /

## 5. Main results
### 5.1 Source-attack success
Applies the success criterion to both methods and summarizes it: TextFooler success before and after the similarity and judge checks, success counts, LLM success rates, queries per attack, the share of TextFooler successes passing each similarity threshold, and 8 random LLM successes for inspection.

**Output:** summary tables and example rewrites.

In [8]:
thr = CFG['sim_threshold']
tf, ll = load('textfooler'), load('llm')
for d in (tf, ll):
    d['judge_ok'] = [bool(judge1(a) == l) if f else False for a, l, f in zip(d.adv, d.label, d.fooled_source)]
    d['sim_ok'] = d.fooled_source & (d.sim.fillna(0) >= thr)
    d['success'] = d.sim_ok & d.judge_ok
save_judge()

print(f'Judge agreement with gold on original sentences: {jacc:.3f}')
print('\nTextFooler source success: raw / after similarity / after similarity + sentiment judge')
print(pd.DataFrame({'raw': tf.groupby('source').fooled_source.mean(),
                    'sim': tf.groupby('source').sim_ok.mean(),
                    'sim+judge': tf.groupby('source').success.mean()}).round(3))
print('Share of TextFooler sim-passing successes the judge says REVERSED the sentiment:',
      round(float(1 - tf[tf.sim_ok].judge_ok.mean()), 3))
print('Success counts -> LLM:', ll.groupby('source').success.sum().to_dict(), '| TextFooler:', tf.groupby('source').success.sum().to_dict())
print('\nLLM source success (similarity + judge inside the loop):')
print(ll.groupby('source').success.mean().round(3))
print('LLM candidates rejected by the judge (fooled + similar but sentiment reversed), mean per sentence:',
      round(float(ll.n_flipped.mean()), 2))
print('LLM avg classifier queries:', ll.groupby('source').queries.mean().round(1).to_dict(),
      '| avg LLM calls:', round(float(ll.llm_calls.mean()), 2))
print('\nShare of TextFooler successes passing each similarity threshold (before judge):')
s = tf[tf.fooled_source].sim
for th in [0.6, 0.7, 0.75, 0.8, 0.85, 0.9]:
    print(' ', th, round(float((s >= th).mean()), 3))
print('\nRandom LLM successes -- read these yourself and check the sentiment is really preserved:')
for _, r in ll[ll.success].sample(min(8, int(ll.success.sum())), random_state=0).iterrows():
    print(f"[{r.source}] sim={r.sim:.2f}\n  orig: {r.text}\n  adv : {r.adv}")

Judge agreement with gold on original sentences: 0.960

TextFooler source success: raw / after similarity / after similarity + sentiment judge
              raw    sim  sim+judge
source                             
BERT        0.945  0.630      0.440
DistilBERT  0.955  0.725      0.575
RoBERTa     0.960  0.635      0.440
Share of TextFooler sim-passing successes the judge says REVERSED the sentiment: 0.269
Success counts -> LLM: {'BERT': 24, 'DistilBERT': 31, 'RoBERTa': 21} | TextFooler: {'BERT': 88, 'DistilBERT': 115, 'RoBERTa': 88}

LLM source success (similarity + judge inside the loop):
source
BERT          0.120
DistilBERT    0.155
RoBERTa       0.105
Name: success, dtype: float64
LLM candidates rejected by the judge (fooled + similar but sentiment reversed), mean per sentence: 0.17
LLM avg classifier queries: {'BERT': 46.3, 'DistilBERT': 44.8, 'RoBERTa': 46.3} | avg LLM calls: 9.25

Share of TextFooler successes passing each similarity threshold (before judge):
  0.6 0.925
  0.7 

### 5.2 Per-pair transfer rates
For each source-to-target pair and method, computes the transfer rate (share of source-successful examples that also fool the target) and the LLM minus TextFooler difference, with 95% bootstrap intervals (2000 resamples of sentences). A pair is marked estimable if both methods have at least 10 source successes. Also reports unconditional rates (examples that fool both models, out of all sentences), success rates and queries per success.

**Output:** transfer table (saved as `main_transfer_table.csv`), success rates, query counts.

In [9]:
df = pd.concat([load('textfooler'), load('llm')], ignore_index=True)
for t in CLF:
    if 'fool_' + t not in df: df['fool_' + t] = np.nan
thr = CFG['sim_threshold']
df['judge_ok'] = [bool(judge1(a) == l) if f else False for a, l, f in zip(df.adv, df.label, df.fooled_source)]
save_judge()
df['success'] = df.fooled_source & (df.sim.fillna(0) >= thr) & df.judge_ok
sids = np.array(sorted(df.sid.unique()))

def arrays(m, s, t):
    d = df[(df.method == m) & (df.source == s)].drop_duplicates('sid').set_index('sid').reindex(sids)
    succ = d.success.eq(True).values
    fool = succ & d['fool_' + t].eq(True).values
    return succ.astype(float), fool.astype(float)

import warnings; warnings.filterwarnings('ignore', category=RuntimeWarning)
MIN_SRC = 10  # minimum source successes for a pair to be estimable
rng = np.random.RandomState(0); B = 2000
rows = []
for s in CLF:
    for t in CLF:
        if s == t: continue
        a = {m: arrays(m, s, t) for m in ['textfooler', 'llm']}
        idx = rng.randint(0, len(sids), (B, len(sids)))
        full = np.arange(len(sids))
        def rate(m, ix):  # NaN if a method has no source successes
            num, den = a[m][1][ix].sum(-1), a[m][0][ix].sum(-1)
            return np.where(den > 0, num / np.maximum(den, 1), np.nan)
        d = rate('llm', idx) - rate('textfooler', idx)
        ci = lambda m: np.nanpercentile(rate(m, idx), [2.5, 97.5]).round(3).tolist()
        rows.append(dict(pair=f'{s}->{t}', estimable=bool(a['textfooler'][0].sum() >= MIN_SRC and a['llm'][0].sum() >= MIN_SRC),
            n_src_tf=int(a['textfooler'][0].sum()), n_src_llm=int(a['llm'][0].sum()),
            transfer_tf=round(float(rate('textfooler', full)), 3), ci_tf=ci('textfooler'),
            transfer_llm=round(float(rate('llm', full)), 3), ci_llm=ci('llm'),
            diff=round(float(rate('llm', full) - rate('textfooler', full)), 3),
            diff_lo=round(float(np.nanpercentile(d, 2.5)), 3), diff_hi=round(float(np.nanpercentile(d, 97.5)), 3),
            uncond_tf=round(float(a['textfooler'][1].sum() / len(sids)), 3),
            uncond_llm=round(float(a['llm'][1].sum() / len(sids)), 3)))
res = pd.DataFrame(rows)
print(res.to_string(index=False))
e = res[res.estimable]
print(f'\nEstimable pairs (>= {MIN_SRC} source successes for both methods): {len(e)} of 6')
print('Of those, LLM > TextFooler with the 95% CI excluding 0:', int(((e['diff'] > 0) & (e.diff_lo > 0)).sum()))
print('Source success rate (similarity + sentiment judge):')
print(df.groupby(['method', 'source']).success.mean().round(3).unstack())
print('Avg queries per successful example:')
print(df[df.success].groupby(['method', 'source']).queries.mean().round(1).unstack())
res.to_csv(f'{OUT}/{CFG["mode"]}_transfer_table.csv', index=False)

               pair  estimable  n_src_tf  n_src_llm  transfer_tf          ci_tf  transfer_llm         ci_llm  diff  diff_lo  diff_hi  uncond_tf  uncond_llm
      BERT->RoBERTa       True        88         24        0.273 [0.184, 0.371]         0.375 [0.174, 0.571] 0.102   -0.123    0.317      0.120       0.045
   BERT->DistilBERT       True        88         24        0.477 [0.373, 0.584]         0.625  [0.419, 0.81] 0.148   -0.063    0.354      0.210       0.075
      RoBERTa->BERT       True        88         21        0.409 [0.305, 0.512]         0.429 [0.211, 0.647] 0.019   -0.218    0.259      0.180       0.045
RoBERTa->DistilBERT       True        88         21        0.295   [0.2, 0.392]         0.524  [0.312, 0.75] 0.228   -0.004    0.468      0.130       0.055
   DistilBERT->BERT       True       115         31        0.348 [0.264, 0.438]         0.387 [0.214, 0.571] 0.039   -0.146    0.240      0.200       0.060
DistilBERT->RoBERTa       True       115         31        0.148

### 5.3 Edit size and hand-check sample
Compares edit ratio (1 minus the word-level similarity ratio) and embedding similarity between methods, and transfer rates within edit-size terciles. Then draws 30 successful examples per method for manual checking and writes `handcheck.csv`, `handcheck_blind.csv` (shuffled, without the method) and `handcheck_key.csv` (method and source for each row). Finally prints 10 random attack outcomes, including failures.

**Output:** edit-size statistics, transfer by edit-size bin, random examples.

Re-running this cell regenerates the three hand-check files and overwrites any saved labels.

In [10]:
def edit_ratio(a, b):
    return 1 - difflib.SequenceMatcher(None, a.split(), b.split()).ratio()

ok = df[df.success].copy()
ok['edit'] = [edit_ratio(a, b) for a, b in zip(ok.text, ok.adv)]
ok['transfer'] = ok[['fool_' + t for t in CLF]].astype(float).mean(axis=1)  # mean over the 2 targets
print('Edit ratio and similarity by method:')
print(ok.groupby('method')[['edit', 'sim']].describe().T.round(3))
ok['bin'] = pd.qcut(ok.edit.rank(method='first'), 3, labels=['small', 'mid', 'large'])
print('\nTransfer rate within edit-size bins (only bins containing BOTH methods can be compared):')
print(ok.groupby(['bin', 'method'], observed=True).transfer.agg(['mean', 'count']).round(3))

chk = pd.concat([ok[ok.method == m].sample(min(30, int((ok.method == m).sum())), random_state=0)
                 for m in ['textfooler', 'llm']])
chk[['method', 'source', 'text', 'adv', 'sim']].assign(my_label='').to_csv(f'{OUT}/handcheck.csv', index=False)
print('\nSaved handcheck.csv: fill my_label with pos/neg for each adv. Compare with the judge, and report the preservation rate per method.')

chk = chk.sample(frac=1, random_state=0)
chk[['text', 'label', 'adv']].assign(my_label='').to_csv(f'{OUT}/handcheck_blind.csv')
chk[['method', 'source']].to_csv(f'{OUT}/handcheck_key.csv')  # open only after labeling

print('\nRandom examples, including failures:')
for _, r in df.sample(10, random_state=1).iterrows():
    print(f"[{r.method}/{r.source}] success={r.success}  sim={r.sim}\n  orig: {r.text}\n  adv : {r.adv}")

Edit ratio and similarity by method:
method         llm  textfooler
edit count  76.000     291.000
     mean    0.797       0.103
     std     0.093       0.059
     min     0.556       0.024
     25%     0.744       0.053
     50%     0.804       0.091
     75%     0.869       0.132
     max     1.000       0.276
sim  count  76.000     291.000
     mean    0.846       0.909
     std     0.030       0.054
     min     0.802       0.800
     25%     0.818       0.862
     50%     0.844       0.910
     75%     0.866       0.955
     max     0.946       0.999

Transfer rate within edit-size bins (only bins containing BOTH methods can be compared):
                   mean  count
bin   method                  
small textfooler  0.328    122
mid   textfooler  0.307    122
large llm         0.421     76
      textfooler  0.319     47

Saved handcheck.csv: fill my_label with pos/neg for each adv. Compare with the judge, and report the preservation rate per method.

Random examples, including 

### 5.4 Pooled transfer rate and similarity sensitivity
Pools the six source-to-target pairs. The pooled rate is the number of target-fooling examples divided by twice the number of source successes. Intervals come from a bootstrap over sentences (2000 resamples). Run with similarity thresholds 0.80 and 0.85.

**Output:** one line per threshold.

In [11]:
def pooled(thr_, B=2000, seed=1):
    d = df.copy()
    d['succ'] = d.fooled_source & (d.sim.fillna(0) >= thr_) & d.judge_ok
    d['nfool'] = d[['fool_' + t for t in CLF]].eq(True).sum(axis=1) * d.succ
    g = {m: d[d.method == m].groupby('sid')[['succ', 'nfool']].sum().reindex(sids).fillna(0)
         for m in ['textfooler', 'llm']}
    idx = np.random.RandomState(seed).randint(0, len(sids), (B, len(sids)))
    full = np.arange(len(sids))
    def rate(m, ix):  # 2 targets per source success
        s_, f_ = g[m].succ.values[ix].sum(-1), g[m].nfool.values[ix].sum(-1)
        return np.where(s_ > 0, f_ / np.maximum(2 * s_, 1), np.nan)
    diff = rate('llm', idx) - rate('textfooler', idx)
    tfr, llr = float(rate('textfooler', full)), float(rate('llm', full))
    print(f'thr={thr_}: successes TF={int(g["textfooler"].succ.sum())}, LLM={int(g["llm"].succ.sum())} | '
          f'pooled transfer TF={tfr:.3f}, LLM={llr:.3f} | diff={llr - tfr:+.3f}, '
          f'95% CI [{np.nanpercentile(diff, 2.5):+.3f}, {np.nanpercentile(diff, 97.5):+.3f}]')
for th in [0.80, 0.85]:
    pooled(th)

thr=0.8: successes TF=291, LLM=76 | pooled transfer TF=0.318, LLM=0.421 | diff=+0.103, 95% CI [-0.027, +0.236]
thr=0.85: successes TF=236, LLM=32 | pooled transfer TF=0.309, LLM=0.422 | diff=+0.113, 95% CI [-0.081, +0.290]


## 6. Validity checks
### 6.1 Blind hand labeling
Interactive cell. Shows each sampled rewrite on its own, without the method, the original sentence or the judge's label, and records `p` (positive), `n` (negative) or `u` (unclear) in `handcheck_blind.csv`. Progress is saved after every answer.

**Output:** prompts only; the labels are stored in the CSV.

In [ ]:
import pandas as pd
OUT = '/kaggle/working/hw2_transfer_distilbert_fixed'
lp = f'{OUT}/handcheck_blind.csv'
blind = pd.read_csv(lp, index_col=0)
blind['my_label'] = blind['my_label'].astype('object')
todo = list(blind.index[blind.my_label.isna()])
print(len(todo), 'rewrites left to label')
for k, i in enumerate(todo, 1):
    print(f'\n[{k}/{len(todo)}]')
    print(blind.loc[i, 'adv'])
    a = input('p = positive, n = negative, u = unclear, q = quit: ').strip().lower()
    if a == 'q': break
    if a in ('p', 'n', 'u'):
        blind.loc[i, 'my_label'] = {'p': 'pos', 'n': 'neg', 'u': 'unclear'}[a]
        blind.to_csv(lp)

### 6.2 NLI check
Runs `roberta-large-mnli` between each successful example and its original, in both directions. An example counts as a contradiction if either direction has contradiction probability above 0.5, and as mutually entailing if both directions have entailment probability above 0.5.

**Output:** number of successes, mean entailment, and contradiction and mutual-entailment rates per method.

In [ ]:
import json, numpy as np, pandas as pd, torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
OUT = '/kaggle/working/hw2_transfer_distilbert_fixed'
JC = json.load(open(f'{OUT}/judge_cache.json'))
df2 = pd.concat([pd.read_json(f'{OUT}/main_textfooler.jsonl', lines=True),
                 pd.read_json(f'{OUT}/main_llm_v3.jsonl', lines=True)], ignore_index=True)
df2['ok'] = df2.fooled_source & (df2.sim.fillna(0) >= 0.80) & df2.adv.map(lambda t: JC.get(t)).eq(df2.label)
s = df2[df2.ok].copy()

name = 'roberta-large-mnli'
ntok = AutoTokenizer.from_pretrained(name)
nmdl = AutoModelForSequenceClassification.from_pretrained(name).to('cuda').eval()
ent = [i for i, l in nmdl.config.id2label.items() if l.lower().startswith('entail')][0]
con = [i for i, l in nmdl.config.id2label.items() if l.lower().startswith('contra')][0]

@torch.no_grad()
def nli(prem, hyp, bs=32):
    out = []
    for i in range(0, len(prem), bs):
        enc = ntok(prem[i:i+bs], hyp[i:i+bs], padding=True, truncation=True,
                   max_length=256, return_tensors='pt').to('cuda')
        out.append(torch.softmax(nmdl(**enc).logits, -1).cpu())
    return torch.cat(out).numpy()

fw = nli(list(s.text), list(s.adv))  # original -> rewrite
bw = nli(list(s.adv), list(s.text))  # rewrite -> original
s['ent_fw'], s['ent_bw'] = fw[:, ent], bw[:, ent]
s['contradiction'] = np.maximum(fw[:, con], bw[:, con]) > 0.5
s['bi_entail'] = (s['ent_fw'] > 0.5) & (s['ent_bw'] > 0.5)
print(s.groupby('method').size().rename('n successes'))
print(s.groupby('method')[['ent_fw', 'ent_bw']].mean().round(3))
print(s.groupby('method')[['contradiction', 'bi_entail']].mean().round(3))

method
llm            76
textfooler    291
Name: n successes, dtype: int64
            ent_fw  ent_bw
method                    
llm          0.694   0.718
textfooler   0.810   0.893
            contradiction  bi_entail
method                              
llm                 0.105      0.618
textfooler          0.127      0.838


### 6.3 Pooled transfer on NLI-filtered subsets
Repeats the pooled comparison on all successes, on successes without an NLI contradiction, and on mutually entailing successes.

**Output:** one line per subset.

In [ ]:
sids = np.array(sorted(df2.sid.unique()))
def pooled_subset(sub, name, B=2000, seed=1):
    sub = sub.copy()
    fc = [c for c in sub.columns if c.startswith('fool_')]
    sub['succ'] = 1
    sub['nfool'] = sub[fc].eq(True).sum(axis=1)
    g = {m: sub[sub.method == m].groupby('sid')[['succ', 'nfool']].sum().reindex(sids).fillna(0)
         for m in ['textfooler', 'llm']}
    idx = np.random.RandomState(seed).randint(0, len(sids), (B, len(sids)))
    full = np.arange(len(sids))
    def rate(m, ix):
        s_, f_ = g[m].succ.values[ix].sum(-1), g[m].nfool.values[ix].sum(-1)
        return np.where(s_ > 0, f_ / np.maximum(2 * s_, 1), np.nan)
    d = rate('llm', idx) - rate('textfooler', idx)
    print(f'{name}: n TF={int(g["textfooler"].succ.sum())}, LLM={int(g["llm"].succ.sum())} | '
          f'transfer TF={float(rate("textfooler", full)):.3f}, LLM={float(rate("llm", full)):.3f} | '
          f'diff={float(rate("llm", full) - rate("textfooler", full)):+.3f}, '
          f'95% CI [{np.nanpercentile(d, 2.5):+.3f}, {np.nanpercentile(d, 97.5):+.3f}]')

pooled_subset(s, 'all successes')
pooled_subset(s[~s.contradiction], 'no contradiction')
pooled_subset(s[s.bi_entail], 'mutual entailment')

all successes: n TF=291, LLM=76 | transfer TF=0.318, LLM=0.421 | diff=+0.103, 95% CI [-0.027, +0.236]
no contradiction: n TF=254, LLM=68 | transfer TF=0.303, LLM=0.434 | diff=+0.131, 95% CI [-0.004, +0.273]
mutual entailment: n TF=244, LLM=47 | transfer TF=0.301, LLM=0.479 | diff=+0.177, 95% CI [-0.000, +0.342]


### 6.4 Hand-check results
Joins the labels with the key and reports, per method, how many sampled rewrites kept the original sentiment (strict: unclear counts as not preserved; lenient: clear labels only), how often the judge agrees with the hand label, and mean transfer for the labeled and the preserved examples.

**Output:** counts per method.

In [ ]:
import json, numpy as np, pandas as pd
OUT = '/kaggle/working/hw2_transfer_distilbert_fixed'
blind = pd.read_csv(f'{OUT}/handcheck_blind.csv', index_col=0)
key = pd.read_csv(f'{OUT}/handcheck_key.csv', index_col=0)
hc = blind.join(key)
lab = hc.my_label.astype(str).str.strip().str.lower()
hc['unclear'] = lab.eq('unclear')
hc['my'] = lab.map({'neg': 0, 'pos': 1})
JC = json.load(open(f'{OUT}/judge_cache.json'))
hc['judge'] = hc.adv.map(JC)
df2 = pd.concat([pd.read_json(f'{OUT}/main_textfooler.jsonl', lines=True),
                 pd.read_json(f'{OUT}/main_llm_v3.jsonl', lines=True)], ignore_index=True)
fc = [c for c in df2.columns if c.startswith('fool_')]
hc['transfer'] = df2.loc[hc.index, fc].astype(float).mean(axis=1)
print('unlabeled rows:', int(hc.my.isna().sum() - hc.unclear.sum()))
for m in ['textfooler', 'llm']:
    d = hc[hc.method == m]
    c = d[d.my.notna()]
    p = c[c.my == c.label]
    print(f'\n{m}: n={len(d)}, clear labels={len(c)}, unclear={int(d.unclear.sum())}')
    print(f'  preserved (strict, unclear counts as not preserved): {len(p)}/{len(d)} = {len(p)/len(d):.3f}')
    print(f'  preserved (lenient, clear labels only): {len(p)}/{len(c)} = {len(p)/max(len(c),1):.3f}')
    print(f'  judge agrees with me: {(c.judge == c.my).sum()}/{len(c)}')
    print(f'  mean transfer: all labeled={c.transfer.mean():.3f} | preserved only={p.transfer.mean():.3f}')

unlabeled rows: 0

textfooler: n=30, clear labels=20, unclear=10
  preserved (strict, unclear counts as not preserved): 15/30 = 0.500
  preserved (lenient, clear labels only): 15/20 = 0.750
  judge agrees with me: 15/20
  mean transfer: all labeled=0.475 | preserved only=0.333

llm: n=30, clear labels=26, unclear=4
  preserved (strict, unclear counts as not preserved): 20/30 = 0.667
  preserved (lenient, clear labels only): 20/26 = 0.769
  judge agrees with me: 20/26
  mean transfer: all labeled=0.327 | preserved only=0.325
